In [1]:
# 05c2-1. 기본 설정

from pathlib import Path

import numpy as np
import pandas as pd


PROJECT_ROOT = Path(
    r"C:\code\portfolio_optimization"
)

DATA_ROOT = (
    PROJECT_ROOT
    / "data"
)

RAW_CONTEXT_DIR = (
    DATA_ROOT
    / "raw"
    / "market_context"
)

FEATURE_CONTEXT_DIR = (
    DATA_ROOT
    / "features"
    / "market_context"
)

RAW_CONTEXT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

FEATURE_CONTEXT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


print(
    RAW_CONTEXT_DIR
)

print(
    FEATURE_CONTEXT_DIR
)

C:\code\portfolio_optimization\data\raw\market_context
C:\code\portfolio_optimization\data\features\market_context


In [2]:
# 05c2-2. 현재 market-context inventory

CURRENT_MARKET_FEATURES = [
    "market_return_1d",
    "market_return_5d",
    "market_return_20d",
    "market_volatility_20d",
    "market_drawdown",
    "volume_change_1d",
    "trading_value_change_1d",
    "market_trading_value_ratio_20d"
]

CURRENT_MACRO_FEATURES = [
    "base_rate",
    "usdkrw",
    "bond3y",
    "usdkrw_return_1d",
    "usdkrw_return_5d",
    "usdkrw_return_20d",
    "bond3y_change_1d",
    "bond3y_change_5d",
    "bond3y_change_20d",
    "base_rate_change",
    "rate_spread_3y"
]


print(
    "current market:",
    len(CURRENT_MARKET_FEATURES)
)

print(
    "current macro:",
    len(CURRENT_MACRO_FEATURES)
)

print(
    "current total:",
    len(
        CURRENT_MARKET_FEATURES
        + CURRENT_MACRO_FEATURES
    )
)

current market: 8
current macro: 11
current total: 19


In [3]:
# 05c2-3. 추가 context 명세

CONTEXT_SPEC = pd.DataFrame(
    [
        {
            "group": "us_equity",
            "series": "sp500",
            "source": "FRED",
            "priority": "high",
            "pit_rule": "previous_available_us_close"
        },
        {
            "group": "global_risk",
            "series": "vix",
            "source": "FRED",
            "priority": "high",
            "pit_rule": "previous_available_us_close"
        },
        {
            "group": "us_rate",
            "series": "us2y",
            "source": "FRED",
            "priority": "high",
            "pit_rule": "previous_available_value"
        },
        {
            "group": "us_rate",
            "series": "us10y",
            "source": "FRED",
            "priority": "high",
            "pit_rule": "previous_available_value"
        },
        {
            "group": "kr_flow",
            "series": "foreign_net_buy",
            "source": "KRX",
            "priority": "high",
            "pit_rule": "same_krx_close"
        },
        {
            "group": "kr_flow",
            "series": "institution_net_buy",
            "source": "KRX",
            "priority": "high",
            "pit_rule": "same_krx_close"
        },
        {
            "group": "commodity",
            "series": "wti",
            "source": "FRED",
            "priority": "medium",
            "pit_rule": "previous_available_value"
        },
        {
            "group": "kr_breadth",
            "series": "advance_decline",
            "source": "KRX",
            "priority": "medium",
            "pit_rule": "same_krx_close"
        }
    ]
)


print(
    CONTEXT_SPEC.to_string(
        index=False
    )
)

      group              series source priority                    pit_rule
  us_equity               sp500   FRED     high previous_available_us_close
global_risk                 vix   FRED     high previous_available_us_close
    us_rate                us2y   FRED     high    previous_available_value
    us_rate               us10y   FRED     high    previous_available_value
    kr_flow     foreign_net_buy    KRX     high              same_krx_close
    kr_flow institution_net_buy    KRX     high              same_krx_close
  commodity                 wti   FRED   medium    previous_available_value
 kr_breadth     advance_decline    KRX   medium              same_krx_close


In [4]:
# 05c2-4. fred series 설정

FRED_SERIES = {
    "sp500": "SP500",
    "vix": "VIXCLS",
    "us2y": "DGS2",
    "us10y": "DGS10"
}


FRED_START = "2017-09-01"
FRED_END = "2026-09-15"


print(
    FRED_SERIES
)

{'sp500': 'SP500', 'vix': 'VIXCLS', 'us2y': 'DGS2', 'us10y': 'DGS10'}


In [5]:
# 05c2-5. fred 원본 수집

def load_fred_series(
    series_name,
    series_id,
    start_date,
    end_date
):

    url = (
        "https://fred.stlouisfed.org/"
        "graph/fredgraph.csv"
        f"?id={series_id}"
        f"&cosd={start_date}"
        f"&coed={end_date}"
    )


    data = pd.read_csv(
        url
    )


    data = data.rename(
        columns={
            data.columns[0]: "date",
            data.columns[1]: series_name
        }
    )


    data["date"] = pd.to_datetime(
        data["date"]
    )


    data[series_name] = pd.to_numeric(
        data[series_name],
        errors="coerce"
    )


    return data


fred_raw = {}


for name, series_id in FRED_SERIES.items():

    data = load_fred_series(
        name,
        series_id,
        FRED_START,
        FRED_END
    )

    fred_raw[name] = data

    print(
        name,
        data.shape,
        data["date"].min(),
        data["date"].max(),
        "missing:",
        data[name].isna().sum()
    )

sp500 (2358, 2) 2017-09-01 00:00:00 2026-09-15 00:00:00 missing: 88
vix (2358, 2) 2017-09-01 00:00:00 2026-09-15 00:00:00 missing: 55
us2y (2358, 2) 2017-09-01 00:00:00 2026-09-15 00:00:00 missing: 100
us10y (2358, 2) 2017-09-01 00:00:00 2026-09-15 00:00:00 missing: 100


In [6]:
# 05c2-6. fred raw 저장

for name, data in fred_raw.items():

    path = (
        RAW_CONTEXT_DIR
        / f"fred_{name}_daily.csv"
    )

    data.to_csv(
        path,
        index=False,
        encoding="utf-8-sig"
    )

    print(
        name,
        path
    )

sp500 C:\code\portfolio_optimization\data\raw\market_context\fred_sp500_daily.csv
vix C:\code\portfolio_optimization\data\raw\market_context\fred_vix_daily.csv
us2y C:\code\portfolio_optimization\data\raw\market_context\fred_us2y_daily.csv
us10y C:\code\portfolio_optimization\data\raw\market_context\fred_us10y_daily.csv


US 2026-09-14 close
->
한국 2026-09-15 새벽에 확인 가능
->
한국 2026-09-15 15:30 signal에는 사용 가능

available_date = observation_date + 1 calendar day

미국 9/15 금리 관측
->
미국 9/16 16:15 ET 공개
->
한국 9/17 새벽 공개
->
한국 9/17 signal부터 사용 가능

In [8]:
# 05c2-7. pit 사용가능 날짜 생성

from pandas.tseries.holiday import (
    USFederalHolidayCalendar
)

from pandas.tseries.offsets import (
    CustomBusinessDay
)


US_BUSINESS_DAY = CustomBusinessDay(
    calendar=USFederalHolidayCalendar()
)

ONE_DAY = pd.to_timedelta(
    1,
    unit="D"
)


sp500_pit = (
    fred_raw["sp500"]
    .dropna(
        subset=["sp500"]
    )
    .copy()
)

vix_pit = (
    fred_raw["vix"]
    .dropna(
        subset=["vix"]
    )
    .copy()
)

us2y_pit = (
    fred_raw["us2y"]
    .dropna(
        subset=["us2y"]
    )
    .copy()
)

us10y_pit = (
    fred_raw["us10y"]
    .dropna(
        subset=["us10y"]
    )
    .copy()
)


for data in [
    sp500_pit,
    vix_pit,
    us2y_pit,
    us10y_pit
]:

    data["date"] = pd.to_datetime(
        data["date"]
    )


sp500_pit[
    "available_date"
] = (
    sp500_pit["date"]
    + ONE_DAY
)

vix_pit[
    "available_date"
] = (
    vix_pit["date"]
    + ONE_DAY
)


def add_us_business_days(
    dates,
    days
):

    return pd.Series(
        [
            date
            + days * US_BUSINESS_DAY
            for date in dates
        ],
        index=dates.index
    )


us2y_pit[
    "available_date"
] = add_us_business_days(
    us2y_pit["date"],
    2
)

us10y_pit[
    "available_date"
] = add_us_business_days(
    us10y_pit["date"],
    2
)

In [9]:
# 05c2-7a. pit 날짜 검증

for name, data in {
    "sp500": sp500_pit,
    "vix": vix_pit,
    "us2y": us2y_pit,
    "us10y": us10y_pit
}.items():

    print(
        name,
        "| rows:",
        len(data),
        "| invalid:",
        (
            data["available_date"]
            <= data["date"]
        ).sum(),
        "| missing:",
        data[
            "available_date"
        ].isna().sum()
    )

sp500 | rows: 2270 | invalid: 0 | missing: 0
vix | rows: 2303 | invalid: 0 | missing: 0
us2y | rows: 2258 | invalid: 0 | missing: 0
us10y | rows: 2258 | invalid: 0 | missing: 0


In [10]:
# 05c2-8. pit 날짜 확인

print(
    "sp500"
)

print(
    sp500_pit[
        [
            "date",
            "available_date",
            "sp500"
        ]
    ]
    .tail()
    .to_string(
        index=False
    )
)


print(
    "\nus2y"
)

print(
    us2y_pit[
        [
            "date",
            "available_date",
            "us2y"
        ]
    ]
    .tail()
    .to_string(
        index=False
    )
)

sp500
      date available_date   sp500
2026-09-09     2026-09-10 7636.36
2026-09-10     2026-09-11 7591.70
2026-09-11     2026-09-12 7656.98
2026-09-14     2026-09-15 7619.98
2026-09-15     2026-09-16 7585.73

us2y
      date available_date  us2y
2026-09-09     2026-09-11  4.43
2026-09-10     2026-09-14  4.56
2026-09-11     2026-09-15  4.63
2026-09-14     2026-09-16  4.65
2026-09-15     2026-09-17  4.67


In [11]:
# 05c2-9. s&p500 feature 생성

sp500_features = (
    sp500_pit
    .sort_values("date")
    .copy()
)


sp500_features[
    "sp500_return_1d"
] = (
    sp500_features["sp500"]
    .pct_change(
        periods=1,
        fill_method=None
    )
)

sp500_features[
    "sp500_return_5d"
] = (
    sp500_features["sp500"]
    .pct_change(
        periods=5,
        fill_method=None
    )
)

sp500_features[
    "sp500_return_20d"
] = (
    sp500_features["sp500"]
    .pct_change(
        periods=20,
        fill_method=None
    )
)


print(
    sp500_features[
        [
            "date",
            "available_date",
            "sp500",
            "sp500_return_1d",
            "sp500_return_5d",
            "sp500_return_20d"
        ]
    ]
    .tail()
    .to_string(
        index=False
    )
)

      date available_date   sp500  sp500_return_1d  sp500_return_5d  sp500_return_20d
2026-09-09     2026-09-10 7636.36        -0.004843         0.000641         -0.011884
2026-09-10     2026-09-11 7591.70        -0.005848        -0.009770         -0.020236
2026-09-11     2026-09-12 7656.98         0.008599        -0.011711         -0.018209
2026-09-14     2026-09-15 7619.98        -0.004832        -0.012777         -0.021293
2026-09-15     2026-09-16 7585.73        -0.004495        -0.011441         -0.020572


In [12]:
# 05c2-10. vix feature 생성

vix_features = (
    vix_pit
    .sort_values("date")
    .copy()
)


vix_features[
    "vix_change_1d"
] = (
    vix_features["vix"]
    .diff(1)
)

vix_features[
    "vix_change_5d"
] = (
    vix_features["vix"]
    .diff(5)
)

vix_features[
    "vix_change_20d"
] = (
    vix_features["vix"]
    .diff(20)
)


print(
    vix_features[
        [
            "date",
            "available_date",
            "vix",
            "vix_change_1d",
            "vix_change_5d",
            "vix_change_20d"
        ]
    ]
    .tail()
    .to_string(
        index=False
    )
)

      date available_date   vix  vix_change_1d  vix_change_5d  vix_change_20d
2026-09-09     2026-09-10 16.46           0.74           1.26            1.91
2026-09-10     2026-09-11 17.84           1.38           3.52            3.21
2026-09-11     2026-09-12 15.84          -2.00           1.31            1.59
2026-09-14     2026-09-15 17.10           1.26           1.80            1.91
2026-09-15     2026-09-16 17.20           0.10           1.48            1.36


In [14]:
# 05c2-11. 미국 국채금리 결합
# bp = basis point: 금리 0.01%p

us_rate_features = (
    us2y_pit[
        [
            "date",
            "available_date",
            "us2y"
        ]
    ]
    .merge(
        us10y_pit[
            [
                "date",
                "available_date",
                "us10y"
            ]
        ],
        on="date",
        how="inner",
        suffixes=(
            "_2y",
            "_10y"
        )
    )
)


print(
    "rows:",
    len(us_rate_features)
)

print(
    "availability match:",
    (
        us_rate_features[
            "available_date_2y"
        ]
        ==
        us_rate_features[
            "available_date_10y"
        ]
    ).all()
)

rows: 2258
availability match: True


In [15]:
# 05c2-12. 미국 금리 feature 생성

us_rate_features[
    "available_date"
] = (
    us_rate_features[
        "available_date_2y"
    ]
)


us_rate_features[
    "us2y_change_1d"
] = (
    us_rate_features["us2y"]
    .diff(1)
)

us_rate_features[
    "us2y_change_5d"
] = (
    us_rate_features["us2y"]
    .diff(5)
)

us_rate_features[
    "us10y_change_1d"
] = (
    us_rate_features["us10y"]
    .diff(1)
)

us_rate_features[
    "us10y_change_5d"
] = (
    us_rate_features["us10y"]
    .diff(5)
)


us_rate_features[
    "us_10y_2y_spread"
] = (
    us_rate_features["us10y"]
    -
    us_rate_features["us2y"]
)


print(
    us_rate_features[
        [
            "date",
            "available_date",
            "us2y",
            "us10y",
            "us2y_change_1d",
            "us2y_change_5d",
            "us10y_change_1d",
            "us10y_change_5d",
            "us_10y_2y_spread"
        ]
    ]
    .tail()
    .to_string(
        index=False
    )
)

      date available_date  us2y  us10y  us2y_change_1d  us2y_change_5d  us10y_change_1d  us10y_change_5d  us_10y_2y_spread
2026-09-09     2026-09-11  4.43   4.83            0.04            0.04             0.03             0.04              0.40
2026-09-10     2026-09-14  4.56   4.95            0.13            0.17             0.12             0.16              0.39
2026-09-11     2026-09-15  4.63   4.96            0.07            0.29             0.01             0.19              0.33
2026-09-14     2026-09-16  4.65   4.97            0.02            0.28             0.01             0.19              0.32
2026-09-15     2026-09-17  4.67   5.00            0.02            0.28             0.03             0.20              0.33


In [16]:
# 05c2-13. global context feature 설정

GLOBAL_CONTEXT_FEATURES = [
    "sp500_return_1d",
    "sp500_return_5d",
    "sp500_return_20d",

    "vix",
    "vix_change_1d",
    "vix_change_5d",
    "vix_change_20d",

    "us2y",
    "us2y_change_1d",
    "us2y_change_5d",

    "us10y",
    "us10y_change_1d",
    "us10y_change_5d",

    "us_10y_2y_spread"
]


print(
    "global context features:",
    len(
        GLOBAL_CONTEXT_FEATURES
    )
)

global context features: 14
